# 01 · LangChain 核心概念与实践

> **模块 02 · Agents 框架实践** 的第一篇，从零搭建对 LangChain 的完整认知。

## 你将学到

| 章节 | 内容 |
|------|------|
| § 1 | LangChain 是什么，为什么需要它 |
| § 2 | Chat Models & Prompt Templates —— 和模型对话的标准方式 |
| § 3 | LCEL —— 用 `\|` 操作符串联任意组件 |
| § 4 | Output Parsers —— 把字符串变成结构化数据 |
| § 5 | Tools & Tool Calling —— 让模型调用外部函数 |
| § 6 | Memory —— 多轮对话记忆管理 |
| § 7 | Agent —— 让模型自主决策调用哪个工具 |
| § 8 | RAG with LangChain —— 完整检索增强生成流水线 |

## 环境准备

```bash
pip install langchain langchain-anthropic langchain-community
pip install faiss-cpu sentence-transformers
```

本 Notebook 需要 `ANTHROPIC_API_KEY`（Claude 模型）。  
将 API Key 写入 `.env` 文件：`ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env 文件

# 验证 Key 已加载（不打印具体值）
assert os.getenv('ANTHROPIC_API_KEY'), '请在 .env 文件中设置 ANTHROPIC_API_KEY'
print('✓ API Key 已加载')


---
## § 1 · LangChain 是什么，为什么需要它

### 1.1 直接调用 API 的局限

假设你要做一个「RAG 问答系统」，直接调用 API 需要自己处理每一步，手写大量样板代码：

```mermaid
flowchart TD
    Q[用户问题] --> VEC[向量化查询]
    VEC --> RET[从向量库检索相关文档]
    RET --> PROMPT["拼接 Prompt\n系统提示 + 文档 + 问题"]
    PROMPT --> API[调用 LLM API]
    API --> PARSE["解析输出\n可能是 JSON"]
    PARSE --> RETRY[错误重试]
    RETRY --> STREAM[流式传输给前端]
```

**LangChain 就是把这些常用模式抽象成可复用组件。**

### 1.2 LangChain 的核心抽象

| 组件类别 | 说明 |
|----------|------|
| **Models** 模型 | 封装各家 LLM，统一调用接口 |
| **PromptTemplates** 提示模板 | 参数化模板，复用提示结构 |
| **OutputParsers** 输出解析 | 把模型输出转成结构化数据 |
| **Retrievers** 检索器 | 抽象向量库/BM25/混合搜索 |
| **Document Loaders** 文档加载器 | PDF/网页/代码等统一加载 |
| **Tools + Agents** 工具与智能体 | 让模型调用函数、执行多步推理 |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# 颜色定义
colors = {
    'model': '#4A90D9',
    'prompt': '#7ED321',
    'parser': '#F5A623',
    'retriever': '#9B59B6',
    'tool': '#E74C3C',
    'memory': '#1ABC9C',
    'agent': '#E67E22',
    'chain': '#95A5A6',
}

# 绘制组件方框
components = [
    (1.0, 4.5, 'Chat Models\nClaude/GPT/Gemini', 'model'),
    (4.0, 4.5, 'Prompt Templates\n动态插值', 'prompt'),
    (7.5, 4.5, 'Output Parsers\nJSON/Pydantic', 'parser'),
    (1.0, 2.5, 'Document Loaders\nPDF/Web/CSV', 'retriever'),
    (4.0, 2.5, 'Vector Stores\nFAISS/Chroma', 'retriever'),
    (7.5, 2.5, 'Retrievers\n语义检索', 'retriever'),
    (1.5, 0.8, 'Tools\n函数/API', 'tool'),
    (4.5, 0.8, 'Memory\n对话历史', 'memory'),
    (7.5, 0.8, 'Agents\n自主决策', 'agent'),
]

for x, y, label, ctype in components:
    rect = FancyBboxPatch((x, y), 2.5, 1.0,
                          boxstyle='round,pad=0.05',
                          facecolor=colors[ctype], alpha=0.85, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+1.25, y+0.5, label, ha='center', va='center',
            fontsize=9, color='white', fontweight='bold')

# LCEL 标签
ax.text(6, 5.7, 'LangChain Expression Language (LCEL)',
        ha='center', fontsize=12, fontweight='bold', color='#2C3E50')
ax.text(6, 5.4, 'prompt | model | parser  ← 用 | 操作符串联任意组件',
        ha='center', fontsize=9, color='#7F8C8D')

# 箭头
for x_s, y_s, x_e, y_e in [(2.5, 5.0, 4.0, 5.0), (6.5, 5.0, 7.5, 5.0)]:
    ax.annotate('', xy=(x_e, y_e), xytext=(x_s, y_s),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

plt.title('LangChain 组件体系', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('langchain_arch.png', dpi=120, bbox_inches='tight')
plt.show()
print('LangChain 将所有组件统一为 Runnable 接口，可用 | 无缝串联')


---
## § 2 · Chat Models & Prompt Templates

### 2.1 为什么需要 Prompt Templates？

直接拼字符串会让代码越来越脏：

```python
# ❌ 直接拼接——不可复用、不可测试
prompt = f'请将以下{language}代码翻译成{target}: \n\n{code}'

# ✅ 用模板——可复用、参数清晰、可序列化
template = ChatPromptTemplate.from_messages([...])
```

### 2.2 消息类型

LangChain 把对话消息分为三种：

| 类型 | 对应角色 | 作用 |
|------|----------|------|
| `SystemMessage` | system | 设定 AI 的角色和行为规则 |
| `HumanMessage` | user | 用户的输入 |
| `AIMessage` | assistant | AI 的回复（用于历史记录） |

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 初始化 Claude 模型
# temperature=0 让输出确定性更高，适合需要一致结果的场景
llm = ChatAnthropic(
    model='claude-haiku-4-5-20251001',  # 速度最快的 Claude 模型
    temperature=0,
    max_tokens=1024,
)

# 最简单的调用方式
response = llm.invoke('用一句话解释什么是机器学习')
print(type(response))    # AIMessage
print(response.content)  # 文本内容
print(f'用了 {response.usage_metadata["input_tokens"]} 个输入 token')


In [ ]:
# 构建带参数的 Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一位 {role}，回答要{style}。'),
    ('human', '{question}'),
])

# 查看模板需要哪些参数
print('模板参数:', prompt.input_variables)

# 填充参数，生成具体的消息列表
messages = prompt.invoke({
    'role': '资深数据科学家',
    'style': '简洁准确，避免行话',
    'question': '什么情况下应该用随机森林而不是神经网络？',
})

print('生成的消息：')
for msg in messages.messages:
    print(f'  [{msg.type}]: {msg.content[:80]}...' if len(msg.content) > 80 else f'  [{msg.type}]: {msg.content}')


In [ ]:
# 直接把 prompt 和 llm 串联（先手动展示，后面介绍 LCEL |）
answer = llm.invoke(messages)
print(answer.content)


---
## § 3 · LCEL —— LangChain Expression Language

LCEL 是 LangChain 的「管道语法」，用 Python 的 `|` 运算符串联组件：

```python
chain = prompt | llm | parser
result = chain.invoke({'key': 'value'})
```

### 为什么用 LCEL 而不是直接调用？

| 特性 | 直接调用 | LCEL |
|------|----------|------|
| 流式输出 | 需要手写循环 | `.stream()` 自动支持 |
| 批量处理 | 需要手写并发 | `.batch()` 自动并行 |
| 可观测性 | 无 | 自动记录每步输入/输出 |
| 可序列化 | 不支持 | 可保存成 JSON |

### 工作原理

每个组件都实现了 `Runnable` 协议：
```
Runnable.invoke(input)  → output
Runnable.stream(input)  → Iterator[chunk]
Runnable.batch([inputs]) → [outputs]
```

`|` 操作符实际上创建了一个 `RunnableSequence`，把前一个组件的输出作为后一个的输入。

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# 方式一：直接串联（传统写法需要嵌套调用）
def chain_manual(question):
    messages = prompt.invoke({'role': '助手', 'style': '清晰', 'question': question})
    response = llm.invoke(messages)
    return response.content

# 方式二：LCEL 管道（推荐）
# StrOutputParser 从 AIMessage 中提取 .content 字符串
chain = prompt | llm | StrOutputParser()

# 用法完全一致，但 chain 还额外支持 stream/batch
result = chain.invoke({
    'role': '助手',
    'style': '清晰',
    'question': '解释 LCEL 的 | 操作符',
})
print(result)


In [ ]:
# 流式输出示例
print('=== 流式输出（token 逐个打印）===')
for chunk in chain.stream({
    'role': '助手', 'style': '清晰',
    'question': '用三句话解释什么是 Transformer',
}):
    print(chunk, end='', flush=True)
print()  # 换行


In [ ]:
# 批量调用示例（内部自动并发）
questions = [
    {'role': '助手', 'style': '简洁', 'question': '什么是向量数据库？'},
    {'role': '助手', 'style': '简洁', 'question': '什么是 Embedding？'},
    {'role': '助手', 'style': '简洁', 'question': '什么是 RAG？'},
]

print('=== 批量调用（并发执行）===')
results = chain.batch(questions, config={'max_concurrency': 3})
for q, r in zip(questions, results):
    print(f'Q: {q["question"]}')
    print(f'A: {r[:100]}...' if len(r) > 100 else f'A: {r}')
    print()


---
## § 4 · Output Parsers —— 结构化输出

LLM 的输出是纯文本，但我们往往需要结构化数据（JSON、列表、Pydantic 模型）。
Output Parser 负责把文本变成 Python 对象，同时在 Prompt 里注入格式说明。

### 常用 Parser 类型

| Parser | 输出类型 | 适用场景 |
|--------|----------|----------|
| `StrOutputParser` | `str` | 纯文本输出 |
| `JsonOutputParser` | `dict` | 需要字典/列表 |
| `PydanticOutputParser` | Pydantic 模型 | 带类型验证的结构化输出 |
| `CommaSeparatedListOutputParser` | `list[str]` | 简单列表 |

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List

# 定义期望的输出结构
class MovieReview(BaseModel):
    title: str = Field(description='电影名')
    score: float = Field(description='评分 0-10')
    pros: List[str] = Field(description='优点列表（3条）')
    cons: List[str] = Field(description='缺点列表（2条）')
    summary: str = Field(description='一句话总结')

# JsonOutputParser 会自动在 prompt 里注入格式说明
parser = JsonOutputParser(pydantic_object=MovieReview)

# 查看自动生成的格式说明
print('Parser 注入的格式说明：')
print(parser.get_format_instructions()[:300], '...')


In [ ]:
# 构建完整的结构化输出 chain
review_prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一位专业影评人，请用 JSON 格式给出评测。\n\n{format_instructions}'),
    ('human', '请评测电影：{movie}'),
])

# 把 format_instructions 作为 partial 变量注入（不需要用户传）
review_prompt = review_prompt.partial(
    format_instructions=parser.get_format_instructions()
)

review_chain = review_prompt | llm | parser

result = review_chain.invoke({'movie': '星际穿越（Interstellar）'})
print(type(result))  # dict
print(f"电影：{result['title']}")
print(f"评分：{result['score']}/10")
print('优点：')
for p in result['pros']:
    print(f'  + {p}')
print('缺点：')
for c in result['cons']:
    print(f'  - {c}')
print(f"总结：{result['summary']}")


---
## § 5 · Tools & Tool Calling

### 5.1 什么是 Tool Calling？

LLM 本身不能执行代码、查数据库或调 API。Tool Calling 让模型能够「请求」调用外部函数：

```mermaid
flowchart TD
    U[用户问题] --> M1[模型判断：需要调用工具]
    M1 --> R["模型返回工具请求
{tool: get_weather, args: {...}}"]
    R --> C[你的代码：调用实际 API]
    C --> M2[把结果返回给模型]
    M2 --> ANS[模型生成最终回复]
```

**关键理解**：模型本身不执行工具，它只「请求」调用。真正执行是你的代码。

### 5.2 用 @tool 装饰器定义工具

LangChain 用函数的 **docstring** 作为工具描述，**类型注解** 作为参数 schema，自动生成给模型看的工具说明。

In [ ]:
from langchain_core.tools import tool
import math
import json
from datetime import datetime

# @tool 装饰器把普通函数变成 LangChain Tool
# 函数的 docstring → 模型理解这个工具用来做什么
# 类型注解 → 自动生成 JSON Schema，模型知道传什么参数
@tool
def calculate(expression: str) -> str:
    '''计算数学表达式，支持加减乘除、幂运算和基本数学函数。
    例如: 2**10, sqrt(16), sin(3.14)'''
    try:
        # 只允许安全的数学操作
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        result = eval(expression, {'__builtins__': {}}, allowed)
        return str(result)
    except Exception as e:
        return f'计算错误: {e}'

@tool
def get_current_time(timezone: str = 'Asia/Shanghai') -> str:
    '''获取当前时间，timezone 参数支持标准时区名称。'''
    now = datetime.now()
    return f'当前时间（{timezone}）: {now.strftime("%Y-%m-%d %H:%M:%S")}'

@tool
def search_knowledge_base(query: str, top_k: int = 3) -> str:
    '''在知识库中搜索相关信息。返回最相关的文档片段。'''
    # 模拟知识库（实际应接向量数据库）
    kb = {
        'transformer': 'Transformer 是 2017 年提出的序列模型，完全基于注意力机制，舍弃了 RNN。',
        'bert': 'BERT 是双向 Transformer 编码器，通过 MLM 任务预训练，适合理解任务。',
        'gpt': 'GPT 是 Transformer 解码器，自回归预训练，适合生成任务。',
        'attention': '注意力机制通过计算 Q、K、V 的加权和来捕捉序列中的依赖关系。',
    }
    results = []
    query_lower = query.lower()
    for key, val in kb.items():
        if key in query_lower or query_lower in val.lower():
            results.append(val)
    return '\n'.join(results[:top_k]) if results else '未找到相关信息'

# 查看工具的 JSON Schema（这就是发给模型的工具定义）
print('工具名:', calculate.name)
print('工具描述:', calculate.description)
print('参数 Schema:', json.dumps(calculate.args_schema.schema(), ensure_ascii=False, indent=2))


In [ ]:
# 把工具绑定到模型
tools = [calculate, get_current_time, search_knowledge_base]
llm_with_tools = llm.bind_tools(tools)

# 发送需要工具调用的问题
response = llm_with_tools.invoke('2的32次方等于多少？')
print('模型响应类型:', type(response))
print('是否请求工具调用:', bool(response.tool_calls))

if response.tool_calls:
    for call in response.tool_calls:
        print(f'  工具名: {call["name"]}')
        print(f'  参数: {call["args"]}')
        # 实际执行工具
        result = calculate.invoke(call['args'])
        print(f'  结果: {result}')


---
## § 6 · Memory —— 多轮对话记忆

LLM 本身是无状态的——每次调用都是全新的，它不记得上次说了什么。
**Memory** 解决的就是这个问题：在调用时自动把历史对话注入 prompt。

### 几种常见 Memory 策略

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| Buffer Memory | 存全部历史 | 完整 | 超长对话超出 context |
| Window Memory | 只存最近 N 轮 | 控制长度 | 丢失早期信息 |
| Summary Memory | 旧消息压缩成摘要 | 节省 token | 压缩可能丢细节 |
| Token Memory | 按 token 数量截断 | 精确控制 | 实现复杂 |

### 现代 LangChain 推荐：手动管理历史

LangChain v0.3+ 推荐用 `MessagesPlaceholder` + 手动列表管理历史，
而不是旧版的 `ConversationBufferMemory`（已被标记为遗留 API）。

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

# 构建支持历史的 prompt
chat_prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一位友善的 AI 助手，擅长解释技术概念。'),
    MessagesPlaceholder(variable_name='history'),  # 历史消息占位
    ('human', '{input}'),
])

chat_chain = chat_prompt | llm | StrOutputParser()

# 用列表管理对话历史
history = []

def chat(user_input: str) -> str:
    '''发送一条消息，自动维护历史。'''
    # 调用时把历史注入 prompt
    response = chat_chain.invoke({
        'input': user_input,
        'history': history,
    })
    # 把这轮对话加入历史
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

# 多轮对话演示
print('轮次 1：')
r1 = chat('我想学习 LangChain，从哪里开始？')
print(f'AI: {r1[:200]}...\n')

print('轮次 2（依赖上一轮的上下文）：')
r2 = chat('你上面说的第一步怎么具体操作？')
print(f'AI: {r2[:200]}...\n')

print(f'当前历史消息数: {len(history)} 条')


In [ ]:
# 实现 Window Memory（只保留最近 N 轮）
MAX_HISTORY = 6  # 最多保留 3 轮对话（每轮 = 1 human + 1 ai = 2 条消息）

def chat_with_window(user_input: str, hist: list) -> tuple:
    '''返回 (回复, 更新后的历史)。'''
    # 截断历史到最大长度
    trimmed_hist = hist[-MAX_HISTORY:]

    response = chat_chain.invoke({
        'input': user_input,
        'history': trimmed_hist,
    })

    new_hist = hist + [
        HumanMessage(content=user_input),
        AIMessage(content=response),
    ]
    return response, new_hist

# 可视化 Window Memory 的工作方式
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis('off')

# 模拟 8 轮对话
turns = ['H1', 'A1', 'H2', 'A2', 'H3', 'A3', 'H4', 'A4']
colors_map = ['#3498DB', '#2ECC71'] * 4

for i, (turn, color) in enumerate(zip(turns, colors_map)):
    x = i * 1.1 + 0.5
    in_window = i >= len(turns) - MAX_HISTORY
    alpha = 0.9 if in_window else 0.25
    rect = plt.Rectangle((x, 1.5), 0.9, 1.2, facecolor=color, alpha=alpha, edgecolor='white')
    ax.add_patch(rect)
    ax.text(x + 0.45, 2.1, turn, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

ax.axvline(x=2*1.1+0.5, color='red', linewidth=2, linestyle='--', alpha=0.7)
ax.text(2*1.1+0.3, 3.2, '窗口边界', color='red', fontsize=9)
ax.text(5, 0.8, f'只有最近 {MAX_HISTORY} 条消息（{MAX_HISTORY//2} 轮对话）发送给模型', ha='center', fontsize=10)

blue_p = mpatches.Patch(color='#3498DB', alpha=0.9, label='Human 消息')
green_p = mpatches.Patch(color='#2ECC71', alpha=0.9, label='AI 消息')
gray_p = mpatches.Patch(color='#95A5A6', alpha=0.5, label='截断（不发送）')
ax.legend(handles=[blue_p, green_p, gray_p], loc='upper left', fontsize=9)

plt.title('Window Memory：只保留最近 N 条消息', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('window_memory.png', dpi=120, bbox_inches='tight')
plt.show()


---
## § 7 · Agent —— 让模型自主决策

### Chain vs Agent 的本质区别

```mermaid
flowchart LR
    subgraph CHAIN["Chain 链：路径预先固定"]
        P[Prompt] --> L[LLM] --> PA[Parser]
    end
    subgraph AGENT["Agent 智能体：模型自主决策"]
        M[模型] --> D{决策}
        D -->|调工具| T[执行工具]
        T --> O[观察结果]
        O --> M
        D -->|足够信息| ANS[最终回复]
    end
```

### ReAct 框架

最流行的 Agent 范式是 **ReAct**（Reasoning + Acting）：

```
Thought: 我需要先查一下当前时间...
Action: get_current_time({'timezone': 'Asia/Shanghai'})
Observation: 当前时间（Asia/Shanghai）: 2024-01-15 14:30:00
Thought: 现在我有时间了，可以回答用户...
Action: calculate({'expression': '2**32'})
Observation: 4294967296
Thought: 我有了所有信息，可以给出最终答案了
Final Answer: ...
```

每次循环：**思考（Thought）→ 行动（Action）→ 观察（Observation）**，直到有足够信息给出 Final Answer。

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor

# Agent 的 system prompt 需要告诉模型它有哪些工具，以及如何使用
agent_prompt = ChatPromptTemplate.from_messages([
    ('system', '''你是一位智能助手，可以使用以下工具来回答问题：

- calculate: 计算数学表达式
- get_current_time: 获取当前时间
- search_knowledge_base: 搜索技术知识库

请根据用户的问题，选择合适的工具。如果不需要工具，直接回答。
先思考需要哪些信息，再决定调用哪个工具。'''),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad'),  # 工具调用的中间结果
])

# 创建 agent（负责决策：什么时候调工具，调哪个，传什么参数）
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt,
)

# AgentExecutor 负责执行：调用工具、收集结果、再传给 agent
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,   # 打印每一步的思考和工具调用
    max_iterations=5,  # 防止无限循环
    handle_parsing_errors=True,
)

print('Agent 创建成功，包含工具：', [t.name for t in tools])


In [ ]:
# 测试 Agent 的多步推理
print('=' * 60)
print('问题：现在几点？2的32次方是多少？Transformer 是什么？')
print('=' * 60)

result = agent_executor.invoke({
    'input': '帮我做三件事：1. 告诉我现在的时间；2. 计算 2 的 32 次方；3. 简单介绍一下 Transformer',
    'chat_history': [],
})

print('\n最终答案：')
print(result['output'])


---
## § 8 · 用 LangChain 构建完整 RAG 流水线

LangChain 提供了一套专门针对 RAG 场景的组件，可以快速搭建生产级的检索增强生成系统。

### 完整流水线

```mermaid
flowchart TD
    subgraph IDX["索引阶段（离线处理）"]
        D[文档] --> DL[DocumentLoader]
        DL --> TS[TextSplitter]
        TS --> EM[Embeddings]
        EM --> VS[(VectorStore)]
    end
    subgraph QRY["查询阶段（在线服务）"]
        Q[用户问题] --> QE[Embedding]
        QE --> SR[VectorStore.search]
        VS --> SR
        SR --> PT["PromptTemplate
注入相关文档"]
        PT --> LLM[LLM]
        LLM --> ANS[最终回答]
    end
```

In [ ]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── 1. 准备文档（模拟从 PDF/网页加载）──
docs = [
    Document(
        page_content='''LangChain 是一个用于构建 LLM 应用的框架，提供了一系列可组合的模块。
        核心概念包括：Chain（链）、Agent（智能体）、Tool（工具）、Memory（记忆）、Retriever（检索器）。
        LCEL（LangChain Expression Language）是 LangChain 的声明式组合语法，使用 | 操作符串联组件。
        它支持流式输出、批量处理、异步调用等特性，是 LangChain v0.2+ 的推荐方式。''',
        metadata={'source': 'langchain_intro.txt', 'chapter': 1}
    ),
    Document(
        page_content='''Retrieval-Augmented Generation（RAG）是解决 LLM 知识截止和幻觉问题的主要方案。
        工作原理：将外部文档向量化存储，用户提问时检索最相关的文档片段，拼入 Prompt 让模型作答。
        RAG 的核心挑战包括：文档切块策略、向量化模型选择、检索算法、重排序（Reranking）。
        混合检索（BM25 + 向量搜索）通常比单一方法效果更好。''',
        metadata={'source': 'rag_guide.txt', 'chapter': 2}
    ),
    Document(
        page_content='''LangChain 的 Agent 框架允许 LLM 自主选择调用哪些工具。
        最常用的是 ReAct（Reasoning + Acting）范式：模型交替进行思考（Thought）和行动（Action），
        观察工具返回的结果（Observation），循环直到得出最终答案（Final Answer）。
        AgentExecutor 负责实际执行工具调用，并将结果传回给模型。
        常见工具类型：搜索引擎、计算器、数据库查询、代码执行器、API 调用。''',
        metadata={'source': 'agent_guide.txt', 'chapter': 3}
    ),
]

print(f'文档数量: {len(docs)}')
print(f'第一篇文档字数: {len(docs[0].page_content)} 字')


In [ ]:
# ── 2. 切块 ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,    # 每块最大字符数
    chunk_overlap=30,  # 块间重叠，保证信息连续
    separators=['\n\n', '\n', '。', '，', ' ', ''],
)

chunks = splitter.split_documents(docs)
print(f'切块后数量: {len(chunks)}')
print(f'\n第一个 chunk：')
print(chunks[0].page_content)
print(f'\n来源：{chunks[0].metadata}')


In [ ]:
# ── 3. 向量化 + 存入 FAISS ──
# 使用本地 sentence-transformers（无需 API Key）
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cpu'},
)

# 批量向量化并建立 FAISS 索引
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'向量库建立完成，存储了 {vectorstore.index.ntotal} 个向量')

# 测试检索
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
test_results = retriever.invoke('LCEL 是什么？')
print(f'\n检索到 {len(test_results)} 个相关 chunk：')
for i, doc in enumerate(test_results):
    print(f'  [{i+1}] (来源: {doc.metadata["source"]}) {doc.page_content[:80]}...')


In [ ]:
# ── 4. 构建 RAG Chain ──
def format_docs(docs):
    '''把检索到的文档列表格式化成字符串。'''
    return '\n\n---\n\n'.join([
        f'[来源: {doc.metadata["source"]}]\n{doc.page_content}'
        for doc in docs
    ])

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', '''你是一位技术助手。请基于以下参考文档回答用户的问题。
如果文档中没有相关信息，请说明你不知道，不要凭空编造。

参考文档：
{context}'''),
    ('human', '{question}'),
])

# 用 RunnablePassthrough 把原始 question 透传到 prompt
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 测试 RAG
print('=== RAG 问答测试 ===')
questions = [
    'LCEL 的 | 操作符有什么好处？',
    'RAG 的主要挑战是什么？',
    'Agent 的 ReAct 范式是怎么工作的？',
]

for q in questions:
    print(f'\nQ: {q}')
    answer = rag_chain.invoke(q)
    print(f'A: {answer[:300]}...' if len(answer) > 300 else f'A: {answer}')
    print('-' * 50)


---
## 总结

本 Notebook 覆盖了 LangChain 的核心概念：

| 概念 | 核心作用 | 关键 API |
|------|----------|----------|
| Chat Models | 统一模型调用接口 | `ChatAnthropic`, `.invoke()` |
| Prompt Templates | 可复用的动态提示 | `ChatPromptTemplate.from_messages()` |
| LCEL | 声明式串联组件 | `prompt \| llm \| parser` |
| Output Parsers | 文本 → 结构化数据 | `JsonOutputParser`, `PydanticOutputParser` |
| Tools | 让模型调用外部函数 | `@tool`, `llm.bind_tools()` |
| Memory | 多轮对话历史 | `MessagesPlaceholder` + 手动列表 |
| Agent | 模型自主决策 | `create_tool_calling_agent`, `AgentExecutor` |
| RAG Chain | 检索增强生成 | `FAISS`, `Retriever`, `RunnablePassthrough` |

### 下一步

- **`02_llama_index_rag.ipynb`**：专注 RAG 场景的 LlamaIndex 框架
- **`03_autogen_multiagent.ipynb`**：多 Agent 协作与角色分工
- **`04_claude_agent_sdk.ipynb`**：原生 Claude SDK 工具调用与子 Agent

### 推荐资源

- [LangChain 官方文档](https://python.langchain.com/docs/)
- [LCEL 教程](https://python.langchain.com/docs/expression_language/)
- [LangSmith（调试追踪平台）](https://smith.langchain.com/)